# T07 · Applications en science des données — ADIF72

**Objectifs :**
- Pipeline de modélisation prédictive complet
- Validation des modèles (k-fold, courbes d'apprentissage)
- Biais-variance, sur/sous-apprentissage
- Métriques d'évaluation (RMSE, R², précision, rappel, F1, AUC)
- Étude de cas end-to-end + biais & éthique

**Lien avec le reste :** ce topic assemble T01-T06 en pratique.
Toutes les notions (SVD→PCA, gradient→entraînement, tests→validation)
trouvent ici leur application.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, learning_curve)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                            precision_score, recall_score, f1_score,
                            roc_auc_score)
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt. sklearn importé.")
rng = np.random.default_rng(5)

Prêt. sklearn importé.


## 1 · Pipeline de modélisation régressive (end-to-end)

Les **6 étapes** d'un pipeline classique :
1. **Données** : chargement, exploration
2. **Séparation** train/test (jamais mélanger !)
3. **Prétraitement** : échelle, encodage (fit SUR LE TRAIN SEUL)
4. **Entraînement** du modèle
5. **Évaluation** sur le test (une seule fois)
6. **Déploiement / itération**

**Piège n°1 : data leakage** — le prétraitement (ex. `StandardScaler`)
doit être `fit` sur les données train et `transform` sur le test.
Sinon le modèle « voit » la moyenne du test → évaluation trop optimiste.
En production, on encapsule prétraitement+modèle dans un `Pipeline`.

In [2]:
n = 500
superficie = rng.uniform(50, 200, n)
age = rng.integers(0, 50, n)
nb_pieces = rng.integers(2, 7, n)
prix = (20000 * superficie - 1500 * age + 8000 * nb_pieces
        + rng.normal(0, 30000, n))
X = pd.DataFrame({'superficie': superficie, 'age': age, 'nb_pieces': nb_pieces})
y = pd.Series(prix)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=5)
scaler = StandardScaler().fit(X_tr)          # fit SUR LE TRAIN
X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)

model = LinearRegression().fit(X_tr_s, y_tr)
pred = model.predict(X_te_s)
rmse = np.sqrt(mean_squared_error(y_te, pred))
r2 = r2_score(y_te, pred)
print(f"RMSE = {rmse:,.0f}  (bruit ~30k)")
print(f"R²   = {r2:.4f}")
assert r2 > 0.99 and rmse < 40000

RMSE = 31,396  (bruit ~30k)
R²   = 0.9987


## 2 · Validation k-fold & biais-variance

Le **train/test split** dépend de la chance de la coupe. Le **k-fold
cross-validation** est plus robuste : on coupe les données en `k` parts,
on entraîne `k` fois (à chaque fois sur `k-1` parts, test sur la restante),
puis on moyenne les scores.

**Courbe d'apprentissage** (learning curve) : on varie la taille du train
et on regarde l'écart train/validation.
- Écart grand → **sur-apprentissage** (variance)
- Deux courbes basses → **sous-apprentissage** (biais)
- Courbes qui convergent en haut → bon modèle

In [3]:
# Données non linéaires (sin)
Xnl = rng.uniform(0, 1, (300, 1))
ynl = np.sin(6 * np.pi * Xnl[:, 0]) + rng.normal(0, 0.1, 300)

lin = LinearRegression()
forest = RandomForestRegressor(n_estimators=50, random_state=5)
for name, m in [("Linéaire", lin), ("Forêt", forest)]:
    kf = KFold(n_splits=5, shuffle=True, random_state=5)
    scores = cross_val_score(m, Xnl, ynl, cv=kf, scoring='r2')
    print(f"{name:8s} : R² moyen = {scores.mean():.3f} (±{scores.std():.3f})")

print("\nLe modèle trop simple (linéaire) sous-apprend sur du sin ;")
print("la forêt capture la non-linéarité (compromis biais/variance).")

Linéaire : R² moyen = 0.061 (±0.051)
Forêt    : R² moyen = 0.970 (±0.005)

Le modèle trop simple (linéaire) sous-apprend sur du sin ;
la forêt capture la non-linéarité (compromis biais/variance).


## 3 · Sur-apprentissage explicite (biais-variance)

Le **trade-off** : plus le modèle est flexible, moins il a de biais
mais plus il a de variance.
- **Biais** : erreur due aux hypothèses trop fortes du modèle (sous-apprend)
- **Variance** : sensibilité aux fluctuations de l'échantillon (sur-apprend)
- **MSE = biais² + variance + bruit**

On peut le visualiser avec un polynôme dont on augmente le degré :
au début le modèle est trop rigide (sous-apprend), puis il colle,
puis il épouse le bruit (sur-apprend).

In [4]:
x = rng.uniform(-3, 3, 100)
y = np.cos(x) + rng.normal(0, 0.1, 100)
Xpoly = x.reshape(-1, 1)
Xtr, Xte, ytr, yte = train_test_split(Xpoly, y, test_size=0.3, random_state=5)

print("Degré -> R² train vs R² test :")
for deg in [1, 3, 8, 20]:
    poly = PolynomialFeatures(deg)
    Ptr = poly.fit_transform(Xtr); Pte = poly.transform(Xte)
    m = LinearRegression().fit(Ptr, ytr)
    r2_tr = m.score(Ptr, ytr); r2_te = m.score(Pte, yte)
    print(f"  degré {deg:2d} : train={r2_tr:.3f}  test={r2_te:.3f}"
          f"  {'<-- sur-apprend' if r2_tr - r2_te > 0.3 else ''}")

print("\nBas degré : sous-apprend (biais). Haut degré : sur-apprend (variance).")

Degré -> R² train vs R² test :
  degré  1 : train=0.000  test=-0.071  
  degré  3 : train=0.901  test=0.917  
  degré  8 : train=0.971  test=0.977  
  degré 20 : train=0.724  test=0.692  

Bas degré : sous-apprend (biais). Haut degré : sur-apprend (variance).


## 4 · Classification : métriques

L'**accuracy** est trompeuse en présence de **déséquilibre** :
si 90% des cas sont négatifs, prédire toujours « négatif » donne
accuracy = 0.9 mais F1 ≈ 0. On utilise alors :
- **Précision** : P(déclaré positif | positif) — qualité des alertes
- **Rappel** : P(déclaré positif | vraiment positif) — sensibilité
- **F1** : moyenne harmonique précision/rappel
- **AUC-ROC** : qualité du classement (seuil-invariant) — la plus robuste

La **courbe ROC** trace rappel vs (1-précision) pour tous les seuils ;
AUC = aire sous la courbe.

In [5]:
# Données équilibrées
Xc = rng.normal(0, 1, (400, 2))
Xc[:200] += np.array([2, 0]); Xc[200:] -= np.array([2, 0])
yc = np.array([1]*200 + [0]*200)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=5)
sc = StandardScaler().fit(Xc_tr)
clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xc_tr), yc_tr)
pred = clf.predict(sc.transform(Xc_te))
prob = clf.predict_proba(sc.transform(Xc_te))[:, 1]
acc = accuracy_score(yc_te, pred); prec = precision_score(yc_te, pred)
rec = recall_score(yc_te, pred); f1 = f1_score(yc_te, pred)
auc = roc_auc_score(yc_te, prob)
print(f"Accuracy={acc:.3f}  Précision={prec:.3f}  Rappel={rec:.3f}  "
      f"F1={f1:.3f}  AUC={auc:.3f}")
assert acc > 0.95 and auc > 0.99

# Données déséquilibrées (90/10)
Xd = rng.normal(0, 1, (1000, 2)); yd = np.zeros(1000, int)
Xd[:100] += np.array([3, 0]); yd[:100] = 1
Xdt, Xde, ydt, yde = train_test_split(Xd, yd, test_size=0.3, random_state=5)
scd = StandardScaler().fit(Xdt)
clf2 = LogisticRegression(max_iter=1000).fit(scd.transform(Xdt), ydt)
pred2 = clf2.predict(scd.transform(Xde))
print(f"\nDéséquilibrées : Accuracy={accuracy_score(yde,pred2):.3f} "
      f"(trompeuse !)  F1={f1_score(yde,pred2,zero_division=0):.3f}")
print("-> préférer F1 / AUC à l'accuracy.")

Accuracy=0.967  Précision=0.947  Rappel=0.982  F1=0.964  AUC=0.998

Déséquilibrées : Accuracy=0.977 (trompeuse !)  F1=0.788
-> préférer F1 / AUC à l'accuracy.


## 5 · Biais & considérations éthiques

Un modèle n'est que **aussi bon que ses données**. Les biais classiques :
- **Biais de sélection** : l'échantillon ne représente pas la population
  (ex : seulement les clients actifs). Le modèle est mauvais hors-échantillon.
- **Biais d'historique** : les données encodent les discriminations passées ;
  le modèle les reproduit.
- **Biais de mesure** : la variable cible est mal définie.

**Vérifications systématiques** avant de déployer :
- Couverture des variables de prédiction dans le jeu de test
- Équité (parité démographique, égalité des faux positifs par groupe)
- Sensibilité aux valeurs extrêmes

In [6]:
# Biais de sélection : entraînement sur x>0 seulement
Xfull = rng.uniform(-5, 5, 2000)
yfull = np.sin(Xfull) + rng.normal(0, 0.1, 2000)
mask = Xfull > 0
Xb, yb = Xfull[mask].reshape(-1, 1), yfull[mask]
mb = LinearRegression().fit(Xb, yb)
err_biais = np.mean((mb.predict(Xfull[~mask].reshape(-1,1)) - yfull[~mask])**2)
print(f"Modèle biaisé (entraîné sur x>0) : coef={mb.coef_[0]:.3f}")
print(f"RMSE sur la partie non vue (x<0) = {np.sqrt(err_biais):.3f}")
print("Leçon : échantillon non représentatif = biais systématique.")

Modèle biaisé (entraîné sur x>0) : coef=-0.416
RMSE sur la partie non vue (x<0) = 2.415
Leçon : échantillon non représentatif = biais systématique.


## 6 · Tableau récapitulatif — choisir la bonne méthode


| Situation | Méthode |
|---|---|
| n grande, x ~ normale | tests paramétriques (z, t) |
| n petit / non normal | tests non paramétriques |
| 3+ groupes (moyennes) | ANOVA + Tukey |
| indépendance (catégories) | chi-deux |
| prédiction continue | régression (lin, arbres, forêt) |
| prédiction binaire | régression logistique / arbres |
| données déséquilibrées | F1, AUC (pas l'accuracy) |
| sur-apprentissage | régularisation, plus de données |
| biais d'échantillonnage | stratification, vérif couverture |


In [7]:
print("✅ T07 terminé.")

✅ T07 terminé.
